# Evaluating synthetic data

`model.evaluate()` scores any fitted model on six dimensions and combines them into one weighted composite score:

| Dimension | Weight | Question |
|---|---|---|
| Utility | 35% | Does a classifier trained on synthetic data work on real data? |
| Fidelity | 25% | Do the marginal distributions and correlations match? |
| Privacy | 15% | Does the synthetic data copy real rows? |
| Diversity | 10% | Does it cover the real data's range? |
| Consistency | 10% | Can a classifier tell synthetic rows from real ones? |
| Stability | 5% | Are the scores consistent across sampling seeds? |

In [2]:
from importlib.resources import files
from pathlib import Path

import pandas as pd

from katabatic.artifacts import LocalArtifactStore
from katabatic.models import get_model
from katabatic.pipeline import TrainTestSplitPipeline

car = pd.read_csv(files("katabatic.datasets") / "car.csv")
car.columns = ["buying", "maint", "doors", "persons", "lug_boot", "safety", "class"]
Path("artifacts").mkdir(exist_ok=True)
car.to_csv("artifacts/car.csv", index=False)

model = get_model("naivebayes")
store = LocalArtifactStore("artifacts")
results = TrainTestSplitPipeline(model=model).run(
    input_csv="artifacts/car.csv",
    dataset_name="car",
    artifact_store=store,
    model_name="naivebayes",
    target_column="class",
)
dataset_ref = results["dataset_ref"]
train_df = pd.read_csv(store.open_path(f"{dataset_ref.train_relpath}/train_full.csv"))
test_df = pd.read_csv(store.open_path(f"{dataset_ref.test_relpath}/test_full.csv"))

Loaded data with shape: (1728, 7)
Train label distribution:
 class
unacc    0.700434
acc      0.222142
good     0.039797
vgood    0.037627
Name: proportion, dtype: float64
Test label distribution:
 class
unacc    0.699422
acc      0.222543
good     0.040462
vgood    0.037572
Name: proportion, dtype: float64
Saved dataset artifact under datasets/car/split-20260925-114927
[NaiveBayes] Synthetic data saved to: artifacts/models/naivebayes_car_train-20260925-114927/synthetic

Results saved to: artifacts/evaluations/naivebayes_car_train-20260925-114927/tstr_report.csv

TSTR Evaluation Results:

LR:
Accuracy: 0.8584
F1 Score: 0.8513

MLP:
Accuracy: 0.8064
F1 Score: 0.7932

RF:
Accuracy: 0.7919
F1 Score: 0.7796

XGBoost:
Accuracy: 0.7861
F1 Score: 0.7783


## All six dimensions

Pass the training data the model learned from. `test_data` is the held-out real data used for the utility dimension. The report is also saved as JSON and CSV when you pass `output_dir=`.

In [3]:
report = model.evaluate(train_df, target_col="class", test_data=test_df)
pd.Series(report.dimension_scores, name="score")

[WARNING] categorical_cols and/or continuous_cols not provided. Column types will be auto-detected from dtypes — this may be inaccurate for integer-encoded categorical columns. Pass them explicitly for reliable results.

Running fidelity evaluation...
[WARNING] categorical_cols and continuous_cols were not provided. Auto-detecting from dtypes — integer-encoded categorical columns will be misclassified as continuous. Pass column types explicitly for accurate results.

=== Fidelity Evaluation ===
Overall fidelity score: 0.9899

Categorical JSD (lower = better)  ->  score: 0.9899
  buying                         JSD = 0.0051
  maint                          JSD = 0.0134
  doors                          JSD = 0.0080
  persons                        JSD = 0.0164
  lug_boot                       JSD = 0.0045
  safety                         JSD = 0.0129
  avg                            JSD = 0.0101

Running utility evaluation...

=== Utility Evaluation ===
Overall utility score: 0.9039

Clas

fidelity       0.9899
utility        0.9039
diversity      0.9994
privacy        0.4563
consistency    0.8781
stability      0.9870
Name: score, dtype: float64

In [4]:
report.composite_score

0.8694

## A subset of dimensions

Column types are detected from dtypes. Pass `categorical_cols=` and `continuous_cols=` when categories are stored as integers, since those would otherwise be treated as continuous.

In [5]:
subset = model.evaluate(
    train_df, target_col="class", test_data=test_df, dimensions=["utility", "privacy"]
)
subset.dimension_scores

[WARNING] categorical_cols and/or continuous_cols not provided. Column types will be auto-detected from dtypes — this may be inaccurate for integer-encoded categorical columns. Pass them explicitly for reliable results.

Running utility evaluation...

=== Utility Evaluation ===
Overall utility score: 0.9033

Classifier   Metric     TSTR mean    TRTR mean    Delta   
--------------------------------------------------------
LR           accuracy   0.7052       0.6884       -0.0168
LR           f1         0.6206       0.6097       -0.0109
DT           accuracy   0.7809       0.9734       0.1925
DT           f1         0.7706       0.9730       0.2024
RF           accuracy   0.8197       0.9671       0.1474
RF           f1         0.8083       0.9668       0.1585
LinearSVM    accuracy   0.7127       0.7023       -0.0104
LinearSVM    f1         0.6232       0.6239       0.0007
MLP          accuracy   0.8237       0.9717       0.148
MLP          f1         0.8158       0.9717       0.1559

R

{'utility': 0.9033, 'privacy': 0.4568}

## Your own evaluation

Pass any object with a `run()` method as `pipeline=`, as scikit-learn takes a `scoring=` argument. `evaluate()` still samples the synthetic data and aligns its columns, and returns whatever `run()` returns.

In [6]:
class CategoryCoverage:
    """Share of each column's real categories that appear in the synthetic data."""

    def run(self, real_data, synthetic_data, **kwargs):
        return {
            col: synthetic_data[col].nunique() / real_data[col].nunique()
            for col in real_data.columns
        }


model.evaluate(train_df, pipeline=CategoryCoverage())

{'buying': 1.0,
 'maint': 1.0,
 'doors': 1.0,
 'persons': 1.0,
 'lug_boot': 1.0,
 'safety': 1.0,
 'class': 1.0}